<a href="https://colab.research.google.com/github/alexandrumoldovan1/housing-prices-ml/blob/main/notebooks/04b_per_borough_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Per-borough modeling experiments
# Goal: Test if separate models per borough improve R²

# Imports
import pandas as pd
import numpy as np
import warnings
import time
import joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/ColabProjects/housing-prices-ml'
PROCESSED_DATA_PATH = f'{DRIVE_PATH}/processed_data'
MODELS_PATH = f'{DRIVE_PATH}/models'
OUTPUTS_PATH = f'{DRIVE_PATH}/outputs'

# Load v2 data (with PLUTO + distance features)
X_train = pd.read_parquet(f'{PROCESSED_DATA_PATH}/X_train_v2.parquet')
X_test = pd.read_parquet(f'{PROCESSED_DATA_PATH}/X_test_v2.parquet')
y_train = pd.read_parquet(f'{PROCESSED_DATA_PATH}/y_train_v2.parquet').squeeze()
y_test = pd.read_parquet(f'{PROCESSED_DATA_PATH}/y_test_v2.parquet').squeeze()

print(f"Data loaded:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")

# Identify borough columns (one-hot encoded)
borough_cols = [c for c in X_train.columns if c.startswith('borough_')]
print(f"\nBorough columns: {borough_cols}")

# Check distribution per borough
print(f"\nTrain distribution per borough:")
for col in borough_cols:
    n = X_train[col].sum()
    print(f"  {col}: {n:,} samples ({n/len(X_train)*100:.1f}%)")

Mounted at /content/drive
Data loaded:
  X_train: (99880, 43)
  X_test:  (53522, 43)

Borough columns: ['borough_bronx', 'borough_brooklyn', 'borough_manhattan', 'borough_queens', 'borough_staten_island']

Train distribution per borough:
  borough_bronx: 7,832 samples (7.8%)
  borough_brooklyn: 25,805 samples (25.8%)
  borough_manhattan: 26,589 samples (26.6%)
  borough_queens: 30,267 samples (30.3%)
  borough_staten_island: 9,387 samples (9.4%)


In [2]:
# ============================================
# Per-Borough Modeling Function
# ============================================

def get_borough_mask(df, borough_col):
    """Returns boolean mask for rows in this borough"""
    return df[borough_col] == 1

def train_per_borough(model_class, model_name, model_params, X_tr, y_tr, X_te, y_te, borough_cols):
    """
    Train one model per borough and combine predictions.
    Returns combined predictions on test set.
    """
    print(f"\n{'='*60}")
    print(f"Training {model_name} per borough")
    print(f"{'='*60}")

    # Storage for predictions
    y_pred_combined = np.zeros(len(X_te))
    borough_metrics = {}

    for col in borough_cols:
        borough_name = col.replace('borough_', '')

        # Get train and test for this borough
        train_mask = get_borough_mask(X_tr, col)
        test_mask = get_borough_mask(X_te, col)

        X_tr_b = X_tr[train_mask]
        y_tr_b = y_tr[train_mask]
        X_te_b = X_te[test_mask]
        y_te_b = y_te[test_mask]

        n_train = len(X_tr_b)
        n_test = len(X_te_b)

        # Train borough-specific model
        start = time.time()
        model = model_class(**model_params)
        model.fit(X_tr_b, y_tr_b)
        elapsed = time.time() - start

        # Predict on borough test
        y_pred_b = model.predict(X_te_b)

        # Store predictions in combined array (using test_mask indices)
        test_indices = X_te.index[test_mask]
        y_pred_combined[X_te.index.isin(test_indices)] = y_pred_b

        # Borough-level R²
        r2_b = r2_score(y_te_b, y_pred_b)
        borough_metrics[borough_name] = {
            'n_train': n_train, 'n_test': n_test, 'R2': r2_b, 'time': elapsed
        }

        print(f"  {borough_name:15s}: train={n_train:6,} | test={n_test:6,} | "
              f"R²={r2_b:.4f} | {elapsed:.1f}s")

    # Combined R² on full test set
    r2_combined = r2_score(y_te, y_pred_combined)
    y_pred_dollars = np.expm1(y_pred_combined)
    y_true_dollars = np.expm1(y_te)
    rmse = np.sqrt(mean_squared_error(y_true_dollars, y_pred_dollars))
    mae = mean_absolute_error(y_true_dollars, y_pred_dollars)

    print(f"\n  COMBINED R² (test): {r2_combined:.4f}")
    print(f"  RMSE: ${rmse:,.0f}")
    print(f"  MAE:  ${mae:,.0f}")

    return r2_combined, borough_metrics, y_pred_combined


# Define hyperparameters (same as v2 best)
xgb_params = {
    'n_estimators': 352, 'max_depth': 9, 'learning_rate': 0.0404,
    'subsample': 0.853, 'colsample_bytree': 0.659, 'min_child_weight': 1,
    'n_jobs': -1, 'random_state': 42, 'verbosity': 0
}

lgbm_params = {
    'n_estimators': 464, 'max_depth': 15, 'learning_rate': 0.125,
    'num_leaves': 87, 'subsample': 0.894, 'colsample_bytree': 0.637,
    'n_jobs': -1, 'random_state': 42, 'verbose': -1
}

print("Functions and parameters defined.")
print("Ready to train per-borough models.")

Functions and parameters defined.
Ready to train per-borough models.


In [3]:
# ============================================
# Experiment 1: XGBoost per borough
# ============================================

borough_cols = ['borough_bronx', 'borough_brooklyn', 'borough_manhattan',
                'borough_queens', 'borough_staten_island']

r2_xgb, metrics_xgb, preds_xgb = train_per_borough(
    XGBRegressor, "XGBoost", xgb_params,
    X_train, y_train, X_test, y_test, borough_cols
)


Training XGBoost per borough
  bronx          : train= 7,832 | test= 4,236 | R²=0.2976 | 15.0s
  brooklyn       : train=25,805 | test=13,670 | R²=0.4229 | 12.8s
  manhattan      : train=26,589 | test=15,011 | R²=0.2429 | 12.4s
  queens         : train=30,267 | test=15,888 | R²=0.3803 | 9.5s
  staten_island  : train= 9,387 | test= 4,717 | R²=0.3763 | 9.9s

  COMBINED R² (test): 0.3889
  RMSE: $13,690,770
  MAE:  $2,043,439


In [4]:
# ============================================
# Experiment 2: Outlier Capping at $10M
# ============================================

# Convert log_sale_price back to dollars for filtering
y_train_dollars = np.expm1(y_train)
y_test_dollars = np.expm1(y_test)

# Cap at $10M
CAP = 10_000_000

# How many proprieties are above cap?
above_train = (y_train_dollars > CAP).sum()
above_test = (y_test_dollars > CAP).sum()

print(f"Properties above ${CAP:,} cap:")
print(f"  Train: {above_train:,} ({above_train/len(y_train)*100:.2f}%)")
print(f"  Test:  {above_test:,} ({above_test/len(y_test)*100:.2f}%)")

# Filter train (remove outliers)
train_mask = y_train_dollars <= CAP
X_train_capped = X_train[train_mask].copy()
y_train_capped = y_train[train_mask].copy()

# Filter test (remove outliers for fair comparison)
test_mask = y_test_dollars <= CAP
X_test_capped = X_test[test_mask].copy()
y_test_capped = y_test[test_mask].copy()

print(f"\nAfter capping:")
print(f"  X_train: {X_train.shape} -> {X_train_capped.shape}")
print(f"  X_test:  {X_test.shape} -> {X_test_capped.shape}")
print(f"  y_train range: ${np.expm1(y_train_capped).min():,.0f} - ${np.expm1(y_train_capped).max():,.0f}")

Properties above $10,000,000 cap:
  Train: 2,197 (2.20%)
  Test:  1,843 (3.44%)

After capping:
  X_train: (99880, 43) -> (97683, 43)
  X_test:  (53522, 43) -> (51679, 43)
  y_train range: $17,310 - $10,000,000


In [6]:
!pip install catboost xgboost lightgbm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [7]:
# ============================================
# Train XGBoost + LightGBM + CatBoost on capped data
# ============================================
from catboost import CatBoostRegressor

# CatBoost params (from v2)
cat_params = {
    'iterations': 498, 'depth': 10, 'learning_rate': 0.094,
    'l2_leaf_reg': 9.67, 'subsample': 0.6,
    'random_state': 42, 'verbose': 0, 'thread_count': -1,
    'bootstrap_type': 'Bernoulli'
}

results_capped = {}

# XGBoost
print("="*60)
print("XGBoost on capped data")
print("="*60)
start = time.time()
xgb_capped = XGBRegressor(**xgb_params)
xgb_capped.fit(X_train_capped, y_train_capped)
y_pred = xgb_capped.predict(X_test_capped)
r2 = r2_score(y_test_capped, y_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_test_capped), np.expm1(y_pred)))
mae = mean_absolute_error(np.expm1(y_test_capped), np.expm1(y_pred))
elapsed = time.time() - start
print(f"  R²:   {r2:.4f}")
print(f"  RMSE: ${rmse:,.0f}")
print(f"  MAE:  ${mae:,.0f}")
print(f"  Time: {elapsed:.1f}s")
results_capped['XGBoost'] = {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'model': xgb_capped}

# LightGBM
print("\n" + "="*60)
print("LightGBM on capped data")
print("="*60)
start = time.time()
lgbm_capped = LGBMRegressor(**lgbm_params)
lgbm_capped.fit(X_train_capped, y_train_capped)
y_pred = lgbm_capped.predict(X_test_capped)
r2 = r2_score(y_test_capped, y_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_test_capped), np.expm1(y_pred)))
mae = mean_absolute_error(np.expm1(y_test_capped), np.expm1(y_pred))
elapsed = time.time() - start
print(f"  R²:   {r2:.4f}")
print(f"  RMSE: ${rmse:,.0f}")
print(f"  MAE:  ${mae:,.0f}")
print(f"  Time: {elapsed:.1f}s")
results_capped['LightGBM'] = {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'model': lgbm_capped}

# CatBoost
print("\n" + "="*60)
print("CatBoost on capped data")
print("="*60)
start = time.time()
cat_capped = CatBoostRegressor(**cat_params)
cat_capped.fit(X_train_capped, y_train_capped)
y_pred = cat_capped.predict(X_test_capped)
r2 = r2_score(y_test_capped, y_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_test_capped), np.expm1(y_pred)))
mae = mean_absolute_error(np.expm1(y_test_capped), np.expm1(y_pred))
elapsed = time.time() - start
print(f"  R²:   {r2:.4f}")
print(f"  RMSE: ${rmse:,.0f}")
print(f"  MAE:  ${mae:,.0f}")
print(f"  Time: {elapsed:.1f}s")
results_capped['CatBoost'] = {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'model': cat_capped}

# Summary
print("\n" + "="*60)
print("CAPPED RESULTS SUMMARY")
print("="*60)
print(f"{'Model':<15} {'R² capped':<12} {'R² v2 (full)':<14} {'Δ':<10}")
v2_results = {'XGBoost': 0.4062, 'LightGBM': 0.4132, 'CatBoost': 0.3941}
for model_name in ['XGBoost', 'LightGBM', 'CatBoost']:
    r2_new = results_capped[model_name]['R2']
    r2_old = v2_results[model_name]
    delta = r2_new - r2_old
    sign = '+' if delta > 0 else ''
    print(f"{model_name:<15} {r2_new:<12.4f} {r2_old:<14.4f} {sign}{delta:.4f}")

XGBoost on capped data
  R²:   0.5623
  RMSE: $933,006
  MAE:  $457,569
  Time: 29.1s

LightGBM on capped data
  R²:   0.5539
  RMSE: $937,430
  MAE:  $463,637
  Time: 11.9s

CatBoost on capped data
  R²:   0.5514
  RMSE: $949,645
  MAE:  $468,454
  Time: 75.0s

CAPPED RESULTS SUMMARY
Model           R² capped    R² v2 (full)   Δ         
XGBoost         0.5623       0.4062         +0.1561
LightGBM        0.5539       0.4132         +0.1407
CatBoost        0.5514       0.3941         +0.1573


In [8]:
# ============================================
# Stacking on Capped Data (Baseline)
# ============================================
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

print("Building Stacking Regressor on capped data...")
print("Base learners: XGBoost, LightGBM, CatBoost (trained on capped data)")
print("Meta-learner: Ridge Regression\n")

base_models = [
    ('xgb', xgb_capped),
    ('lgbm', lgbm_capped),
    ('cat', cat_capped)
]

meta_model = Ridge(alpha=1.0)

stacking_capped = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=3,
    n_jobs=-1
)

print("Training Stacking Regressor...")
start = time.time()
stacking_capped.fit(X_train_capped, y_train_capped)
elapsed = time.time() - start

y_pred = stacking_capped.predict(X_test_capped)
r2 = r2_score(y_test_capped, y_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_test_capped), np.expm1(y_pred)))
mae = mean_absolute_error(np.expm1(y_test_capped), np.expm1(y_pred))

print(f"\n{'='*60}")
print(f"STACKING (Capped) Results:")
print(f"{'='*60}")
print(f"  R²:   {r2:.4f}")
print(f"  RMSE: ${rmse:,.0f}")
print(f"  MAE:  ${mae:,.0f}")
print(f"  Time: {elapsed:.1f}s")

# Compare with previous results
print(f"\nProgression so far:")
print(f"  v1 (with leakage):           R² = 0.4485")
print(f"  v2 (no leakage, full data):  R² = 0.4196")
print(f"  v2 capped (no Optuna):       R² = {r2:.4f}")

results_capped['Stacking'] = {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'model': stacking_capped}

Building Stacking Regressor on capped data...
Base learners: XGBoost, LightGBM, CatBoost (trained on capped data)
Meta-learner: Ridge Regression

Training Stacking Regressor...

STACKING (Capped) Results:
  R²:   0.5673
  RMSE: $934,968
  MAE:  $456,300
  Time: 399.2s

Progression so far:
  v1 (with leakage):           R² = 0.4485
  v2 (no leakage, full data):  R² = 0.4196
  v2 capped (no Optuna):       R² = 0.5673


In [10]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 6.6 MB/s eta 0:00:00


In [11]:
# ============================================
# Optuna 30 trials on XGBoost (capped data)
# ============================================
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_xgb_capped(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 2.0),
    }
    model = XGBRegressor(**params, n_jobs=-1, random_state=42, verbosity=0)
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    from sklearn.model_selection import cross_val_score
    scores = cross_val_score(model, X_train_capped, y_train_capped, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Optuna tuning XGBoost (30 trials, capped data)...")
print("Estimated time: 45-60 minutes\n")

start = time.time()
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb_capped, n_trials=30, show_progress_bar=False)
elapsed = time.time() - start

print(f"\nTime: {elapsed/60:.1f} min")
print(f"Best CV R²: {study_xgb.best_value:.4f}")
print(f"Best params: {study_xgb.best_params}")

# Train final model with best params
print("\nTraining final XGBoost with tuned params...")
xgb_tuned = XGBRegressor(**study_xgb.best_params, n_jobs=-1, random_state=42, verbosity=0)
xgb_tuned.fit(X_train_capped, y_train_capped)

y_pred = xgb_tuned.predict(X_test_capped)
r2 = r2_score(y_test_capped, y_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_test_capped), np.expm1(y_pred)))
mae = mean_absolute_error(np.expm1(y_test_capped), np.expm1(y_pred))

print(f"\nXGBoost Tuned (capped):")
print(f"  R²:   {r2:.4f}")
print(f"  RMSE: ${rmse:,.0f}")
print(f"  MAE:  ${mae:,.0f}")
print(f"  Improvement vs baseline capped: {r2 - 0.5623:+.4f}")

# Save
joblib.dump(xgb_tuned, f'{MODELS_PATH}/xgboost_tuned_capped.pkl')
results_capped['XGBoost_tuned'] = {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'model': xgb_tuned}
print(f"\nSaved to: xgboost_tuned_capped.pkl")

Optuna tuning XGBoost (30 trials, capped data)...
Estimated time: 45-60 minutes


Time: 91.1 min
Best CV R²: 0.6587
Best params: {'n_estimators': 688, 'max_depth': 13, 'learning_rate': 0.014545153721505942, 'subsample': 0.7049753185236596, 'colsample_bytree': 0.6182780505482555, 'min_child_weight': 9, 'reg_alpha': 0.7750684034884514, 'reg_lambda': 1.281573776182423}

Training final XGBoost with tuned params...

XGBoost Tuned (capped):
  R²:   0.5593
  RMSE: $933,476
  MAE:  $456,671
  Improvement vs baseline capped: -0.0030

Saved to: xgboost_tuned_capped.pkl
